<a href="https://colab.research.google.com/github/mariajsalgadoq/mariajsalgadoq/blob/main/Majo_Salgado_Assignment7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tokenizers

# Building a Generative Transformer (Mini-GPT) in TensorFlow

## Introduction

In this notebook, we will build a Large Language Model (LLM) from scratch using TensorFlow and Keras. Specifically, we will implement a **Decoder-only Transformer**, the same architecture behind models like GPT-3 and Llama.

We will train this model on the text of Shakespeare to generate Shakespearean-style prose.

**Learning Objectives:**

1.  Data pipeline construction using `tf.data`.
2.  Mathematical understanding of Self-Attention and Causal Masking.
3.  Implementing Custom Keras Layers.
4.  Training and sampling from a generative model.

## Part 1: Setup and Data Pipeline

First, we need to import our libraries and prepare the hyperparameters. You can, and should change this, as you think about how to make the model better.

  * `BLOCK_SIZE` (Context Window): This is how far back the model can "see". If set to 128, the model uses the previous 128 tokens to predict token 129.
  * `EMBEDDING_DIM`: The size of the vector representing each token. Larger = more capacity to understand nuance.
  * `NUM_HEADS`: We split the embedding vector into pieces. This allows the model to attend to different types of relationships (e.g., one head focuses on grammar, another on rhyme) simultaneously.


In [ ]:
# !pip install tokenizers

import os
import time
import math
import numpy as np
import tensorflow as tf
from typing import Tuple, List, Optional

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# --- Hyperparameters ---
# We define these as constants

# The number of sequences to process in parallel.
# Higher = Faster training, but requires more VRAM.
BATCH_SIZE: int = 64

# The "Context Window". How many previous tokens the model looks at to predict the next one.
# If BLOCK_SIZE=128, the model uses tokens t_0...t_127 to predict t_128.
BLOCK_SIZE: int = 128

# The size of the vector representation for each token.
# Larger = The model can represent more complex concepts.
EMBEDDING_DIM: int = 256

# The number of "Heads" in Multi-Head Attention.
# We split the embedding vector into this many independent parts.
NUM_HEADS: int = 8

# The number of Transformer Blocks stacked on top of each other.
# Deeper models can reason more abstractly.
NUM_LAYERS: int = 6

# Randomly zeros out neurons during training to prevent overfitting.
DROPOUT_RATE: float = 0.1

# The optimizer's step size.
LEARNING_RATE: float = 3e-4

# How many times we iterate over the dataset.
EPOCHS: int = 1

# The size of our BPE vocabulary.
# For a dataset like Shakespeare, 5000 is sufficient. GPT-4 uses ~100k.
VOCAB_SIZE: int = 5000

# Set random seeds for reproducibility
tf.random.set_seed(1337)
np.random.seed(1337)

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


### Downloading the Dataset

We will use the `shakespeare` dataset.

In [ ]:
path_to_file = tf.keras.utils.get_file(
    't8.shakespeare.txt',
    'https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text)} characters')
print(text[:250])

Length of text: 5458199 characters
This is the 100th Etext file presented by Project Gutenberg, and
is presented in cooperation with World Library, Inc., from their
Library of the Future and Shakespeare CDROMS.  Project Gutenberg
often releases Etexts that are NOT placed in the Public


### Training the BPE Tokenizer



Standard neural networks cannot understand strings; they need integers.

  * **Character Level:** "The" -\> `[20, 8, 5]`. (Inefficient, long sequences).
  * **Word Level:** "The" -\> `[1452]`. (Vocabulary becomes too huge).
  * **Byte Pair Encoding (BPE):** BPE iteratively merges the most frequent pair of adjacent bytes. It starts with single characters and builds up to common words. It maximizes the compression of the text data by minimizing the number of tokens required to represent the corpus. It merges frequent adjacent characters. "th" might become one token, "ing" another.

**Implementation:**
We use the `tokenizers` library to learn the vocabulary from our specific text file.


In [ ]:
def train_bpe_tokenizer(corpus: str, vocab_size: int) -> Tokenizer:
    """
    Trains a Byte Pair Encoding tokenizer on the provided text corpus.

    Args:
        corpus: The raw string data to train on.
        vocab_size: The target size of the vocabulary.

    Returns:
        A trained Tokenizer object.
    """
    # The library requires training from a file, so we save our string temporarily.
    with open("temp_corpus.txt", "w", encoding="utf-8") as f:
        f.write(corpus)

    # 1. Initialize the BPE model
    tokenizer = Tokenizer(BPE())

    # 2. Pre-tokenization: How to split the text *before* BPE starts.
    # ByteLevel is standard for GPT models; it handles spaces/punctuation robustly.
    tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

    # 3. Decoder: How to turn numbers back into text.
    tokenizer.decoder = ByteLevelDecoder()

    # 4. Trainer: Configures the learning process.
    # We add special tokens for Padding and Unknown words.
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["<PAD>", "<UNK>"],
        show_progress=True
    )

    # 5. Train the tokenizer on our file
    tokenizer.train(files=["temp_corpus.txt"], trainer=trainer)

    # Cleanup temp file
    if os.path.exists("temp_corpus.txt"):
        os.remove("temp_corpus.txt")

    return tokenizer

# Execute training
tokenizer = train_bpe_tokenizer(text, VOCAB_SIZE)
print(f"Trained Vocab Size: {tokenizer.get_vocab_size()}")

# Test the tokenizer
sample_text = "To be or not to be"
encoded_sample = tokenizer.encode(sample_text)
print(f"Original: '{sample_text}'")
print(f"IDs:      {encoded_sample.ids}")
print(f"Tokens:   {encoded_sample.tokens}")

Trained Vocab Size: 5000
Original: 'To be or not to be'
IDs:      [1353, 148, 367, 171, 133, 148]
Tokens:   ['To', 'Ġbe', 'Ġor', 'Ġnot', 'Ġto', 'Ġbe']


### Build the tf.data Pipeline (5 Points)
We convert the text to integers and create sliding windows for training.

We need to feed the model batches of data $(X, Y)$.
For a Generative Model (Causal Language Model), the target $Y$ is simply input $X$ shifted forward by one time step.

**Example:**

  * Input Sequence: `[How, are, you, doing]`
  * Input $X$: `[How, are, you]`
  * Target $Y$: `[are, you, doing]`

You need to implement the split_input_target logic and creating the batches.

* Input: A sequence of length BLOCK_SIZE + 1.

* Goal: Split into (input, target).

  * input: The sequence from index 0 to -1.

  * target: The sequence from index 1 to end.


In [ ]:
def create_tf_dataset(text: str, tokenizer: Tokenizer, block_size: int, batch_size: int) -> tf.data.Dataset:
    """
    Converts text into a batched TensorFlow dataset for causal modeling.
    """
    # 1. Tokenize the entire text into a single long list of integers
    encoded = tokenizer.encode(text)
    # Convert to numpy array for efficient TF conversion
    all_ids = np.array(encoded.ids, dtype=np.int32)

    # 2. Create a basic dataset from the array
    # This creates a stream of individual numbers: 12, 45, 99...
    dataset = tf.data.Dataset.from_tensor_slices(all_ids)

    # 3. Batch into sequences
    # We need (block_size + 1) because we need the extra token for the label.
    # drop_remainder=True ensures all batches are exactly the same shape.
    sequences = dataset.batch(block_size + 1, drop_remainder=True)

    def split_input_target(sequence: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
        """
        TODO: Implement the splitting logic.
        """
        # --- YOUR CODE HERE ---
        input_text = sequence[:-1]
        target_text = sequence[1:]
        # ----------------------------------
        return input_text, target_text

    # 4. Map the split function to every sequence
    dataset = sequences.map(split_input_target)

    # 5. Shuffle, Batch, and Prefetch
    # shuffle(10000): Keeps a buffer of 10,000 sequences to shuffle from.
    dataset = dataset.shuffle(10000)
    # Batch into groups of BATCH_SIZE (e.g., 64 sequences at once)
    dataset = dataset.batch(batch_size, drop_remainder=True)
    # Prefetch: Prepares the next batch while the GPU processes the current one.
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


# Test your implementation
train_dataset = create_tf_dataset(text, tokenizer, BLOCK_SIZE, BATCH_SIZE)
for x, y in train_dataset.take(1):
    print(f"Input shape: {x.shape}")
    assert x.shape == (BATCH_SIZE, BLOCK_SIZE), "Incorrect Input Shape"
    assert y.shape == (BATCH_SIZE, BLOCK_SIZE), "Incorrect Target Shape"
    print("Task 1 Passed!")


Input shape: (64, 128)
Task 1 Passed!


## The Transformer Architecture (10 Points)

We will now build the model. We use **Subclassing API** (`tf.keras.layers.Layer`) for maximum control.




### Causal Self-Attention (5 points)


$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$$

**Intuition:**

1.  **Query ($Q$):** Each token asks: "What information am I looking for?"
2.  **Key ($K$):** Each token advertises: "What information do I hold?"
3.  **$QK^T$:** We multiply Query and Key. High value = Good match (strong attention).
4.  **Scale ($\sqrt{d_k}$):** We divide by the square root of dimension size to keep numbers small (stable gradients).
5.  **Mask ($M$):** We add $-\infty$ to positions in the future. This forces the Softmax to be 0 for those positions. **This prevents the model from cheating by looking ahead.**
6.  **Value ($V$):** If we pay attention to a token, we aggregate its "Value" vector.


You must implement the `call` method for Causal Self-Attention.
Refer to the equation: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V$

**Requirements:**

1.  Calculate `scores` using matrix multiplication of Q and K.
2.  Scale the `scores` by dividing by $\sqrt{\text{key_dim}}$.
3.  Apply the `mask`: Add `-1e9` to the future positions (where the mask is 0).
4.  Apply Softmax.
5.  Multiply by V.

In [ ]:
class CausalSelfAttention(tf.keras.layers.Layer):
    """
    Implements Multi-Head Causal Self-Attention.
    """
    def __init__(self, num_heads: int, key_dim: int, dropout: float):
        super().__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        # The total depth is split across heads.
        # E.g., if Embed Dim=256 and Heads=4, each head processes 64 dims.
        self.d_model = num_heads * key_dim

        # Linear layers to project input X into Q, K, and V
        self.q_proj = tf.keras.layers.Dense(self.d_model)
        self.k_proj = tf.keras.layers.Dense(self.d_model)
        self.v_proj = tf.keras.layers.Dense(self.d_model)

        # Final linear layer to mix the results of all heads
        self.out_proj = tf.keras.layers.Dense(self.d_model)

        # Dropout layer for regularization
        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        # B = Batch Size, T = Sequence Length (Time)
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]

        # 1. Calculate Query, Key, Value matrices
        # Shape: (B, T, d_model)
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # 2. Split the heads
        # We reshape to (B, T, Heads, Key_Dim)
        q = tf.reshape(q, (B, T, self.num_heads, self.key_dim))
        k = tf.reshape(k, (B, T, self.num_heads, self.key_dim))
        v = tf.reshape(v, (B, T, self.num_heads, self.key_dim))

        # Transpose to (B, Heads, T, Key_Dim) so matrix multiplication
        # occurs on the last two dimensions (T, Key_Dim) independently per head.
        q = tf.transpose(q, perm=[0, 2, 1, 3])
        k = tf.transpose(k, perm=[0, 2, 1, 3])
        v = tf.transpose(v, perm=[0, 2, 1, 3])

        # --- YOUR CODE HERE ---

        # 3. Calculate Attention Scores (Scaled Dot-Product)
        # Matrix Mult: (B, Heads, T, K) @ (B, Heads, K, T) -> (B, Heads, T, T)
        # This gives a T x T matrix of affinity scores for every token pair.
        scores = tf.matmul(q, k, transpose_b=True)

        # Scale by sqrt(key_dim) to prevent exploding gradients
        # Hint: cast key_dim to float32 before sqrt
        scale = tf.cast(self.key_dim, tf.float32) ** 0.5
        scores = scores / scale

        # 4. Apply Causal Mask
        # Create a matrix where the lower triangle is 1 (past) and upper is 0 (future)
        mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        mask = tf.reshape(mask, (1, 1, T, T))

        # We want to replace 0s with -infinity so Softmax makes them 0.
        # We use -1e9 as a proxy for negative infinity.
        # TODO: Apply mask to scores.
        scores = scores + (1.0 - mask) * -1e9

        # Where mask is 0, scores should become -1e9.
        # Where mask is 1, scores should remain unchanged.

        # 5. Softmax -> Probabilities
        # Normalize scores along the last axis so they sum to 1
        attn_weights = tf.nn.softmax(scores, axis=-1)

        # Apply dropout to the attention weights (randomly ignore some connections)
        attn_weights = self.dropout(attn_weights, training=training)

        # 6. Aggregate Values
        # (B, Heads, T, T) @ (B, Heads, T, K) -> (B, Heads, T, K)
        out = tf.matmul(attn_weights, v)

        # 7. Reassemble Heads
        # Transpose back to (B, T, Heads, K)
        out = tf.transpose(out, perm=[0, 2, 1, 3])
        # Flatten Heads and K back into d_model: (B, T, d_model)
        out = tf.reshape(out, (B, T, self.d_model))

        # Final projection
        return self.out_proj(out)

### Feed Forward Network (MLP)

**Intuition:**
After tokens exchange information via Attention, each token needs to "think" about what it learned. The Feed Forward network processes every token **individually**.

**Architecture:**

1.  Expand dimension (usually $4 \times$ embedding size).
2.  Apply Activation (ReLU).
3.  Project back to original dimension.




In [ ]:
class FeedForward(tf.keras.layers.Layer):
    """
    Point-wise Feed Forward Network.
    """
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.net = tf.keras.Sequential([
            # Expand: Inner layer is 4x wider
            tf.keras.layers.Dense(4 * d_model, activation='relu'),
            # Contract: Project back to d_model
            tf.keras.layers.Dense(d_model),
            # Dropout for regularization
            tf.keras.layers.Dropout(dropout)
        ])

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        return self.net(x, training=training)


### The Transformer Block


A block combines Attention and FeedForward with two crucial tricks:

1.  **Residual Connections (Skip Connections):** $x + \text{Layer}(x)$. This creates a "highway" for gradients to flow through deep networks. Here is a good video to learn more about Residual Connections [Youtube](https://www.youtube.com/watch?v=6jucq_mdhdE&t=194s)
2.  **Layer Normalization:** Keeps the inputs to each layer centered (mean 0, variance 1).

We will combine the Attention layer and the FeedForward layer.


In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
    """
    A single block of the Transformer architecture.
    """
    def __init__(self, num_heads: int, key_dim: int, d_model: int, dropout: float):
        super().__init__()
        self.sa = CausalSelfAttention(num_heads, key_dim, dropout)
        self.ffwd = FeedForward(d_model, dropout)

        # Layer Norms (one before attention, one before feed-forward)
        self.ln1 = tf.keras.layers.LayerNormalization()
        self.ln2 = tf.keras.layers.LayerNormalization()

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:
        # We use the "Pre-Norm" formulation (standard in modern GPTs)

        # 1. Normalize -> Attention -> Add Residual
        x = x + self.sa(self.ln1(x), training=training)

        # 2. Normalize -> FeedForward -> Add Residual
        x = x + self.ffwd(self.ln2(x), training=training)

        return x


### The Full GPT Model (5 Points)


1.  **Embeddings:** Convert Token IDs to vectors.
2.  **Positional Embeddings:** Since Attention has no concept of "order", we add a learnable vector for Position 0, Position 1, etc.
3.  **Stack of Blocks:** The main processing engine.
4.  **Output Head:** Converts the final vector back into probabilities over the vocabulary.


In [ ]:
class GPT(tf.keras.Model):
    """
    The complete GPT Architecture.
    """
    def __init__(self, vocab_size: int, num_layers: int, num_heads: int,
                 d_model: int, block_size: int, dropout: float):
        super().__init__()
        self.block_size = block_size

        # 1. Token Embedding Layer: Maps integer IDs to vectors
        self.token_embedding = tf.keras.layers.Embedding(vocab_size, d_model)

        # 2. Positional Embedding Layer: Maps position indices [0, 1, 2...] to vectors
        self.position_embedding = tf.keras.layers.Embedding(block_size, d_model)

        # 3. The stack of Transformer Blocks
        self.blocks = [
            TransformerBlock(num_heads, d_model // num_heads, d_model, dropout)
            for _ in range(num_layers)
        ]

        # 4. Final Layer Norm
        self.ln_f = tf.keras.layers.LayerNormalization()

        # 5. Language Model Head (Output Projection)
        # Projects from d_model back to vocab_size to get logits
        self.lm_head = tf.keras.layers.Dense(vocab_size)

    def call(self, idx: tf.Tensor, training: bool = False) -> tf.Tensor:
        # idx shape: (Batch_Size, Time)
        B = tf.shape(idx)[0]
        T = tf.shape(idx)[1]

        # --- YOUR CODE HERE ----
        # 1. Get Token Embeddings
        # Shape: (B, T, d_model)
        tok_emb = self.token_embedding(idx)


        # 2. Get Position Embeddings
        # Create a range [0, 1, ... T-1]
        pos_ids = tf.range(T)
        pos_emb = self.position_embedding(pos_ids)

        # Add them (x = tok + pos)
        #  (Broadcasting applies pos_emb to every batch item)
        x = tok_emb + pos_emb

        # 3. Pass x through the Transformer Blocks
        for block in self.blocks:
            x = block(x, training=training)

        # 4. Final Norm
        x = self.ln_f(x)

        # 5. Calculate Logits
        logits = self.lm_head(x) # Shape: (B, T, vocab_size)

        return logits

## Training the Model (5 Points)

We use a custom training loop for maximum transparency.

  * **Optimizer:** `AdamW` (Adam with Weight Decay). This is the standard for training Transformers.
  * **Loss Function:** `SparseCategoricalCrossentropy`. "Sparse" because our targets are integers, not one-hot vectors. "Logits=True" because our model outputs raw scores, not Softmax probabilities.

  Implement the calculation of gradients and the application of updates to the weights.


In [ ]:
# 1. Instantiate the Model
model = GPT(
    vocab_size=tokenizer.get_vocab_size(),
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    d_model=EMBEDDING_DIM,
    block_size=BLOCK_SIZE,
    dropout=DROPOUT_RATE
)

# 2. Learning Rate Schedule
# We decay the learning rate over time to help the model settle in the minimum.
# CosineDecay is a popular choice.
num_train_steps = len(train_dataset) * EPOCHS
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=num_train_steps,
    alpha=0.1 # Minimum LR will be 10% of initial
)

# 3. Optimizer
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule)

# 4. Loss Function
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# 5. Training Step Function
# @tf.function compiles this into a TensorFlow Graph for performance.
@tf.function
def train_step(inputs: tf.Tensor, targets: tf.Tensor) -> tf.Tensor:
    with tf.GradientTape() as tape:
        # Forward pass (training=True enables Dropout)
        logits = model(inputs, training=True)

        # Compute loss
        loss = loss_fn(targets, logits)

    # --- YOUR CODE HERE ---
    # 1. Calculate gradients of 'loss' w.r.t. 'model.trainable_variables'
    gradients = tape.gradient(loss, model.trainable_variables)

    # 2. Apply gradients using 'optimizer.apply_gradients'
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    # --------------------------------
    return loss

# --- Main Training Loop ---
print(f"Starting training for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    start_time = time.time()
    epoch_loss = 0.0
    num_batches = 0

    # Iterate over the dataset
    for batch_inputs, batch_targets in train_dataset:
        loss = train_step(batch_inputs, batch_targets)
        epoch_loss += float(loss)
        num_batches += 1

    avg_loss = epoch_loss / num_batches
    duration = time.time() - start_time

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {duration:.2f}s")

Starting training for 1 epochs...


## Inference with Temperature (5 Points)

To generate text, we cannot simply call `model(input)`. We must generate **autoregressively** (one token at a time).

**Algorithm:**

1.  Take input sequence (e.g., "To be").
2.  Get model predictions for the next token.
3.  Apply **Temperature**:
      * Scaling logits by a number $< 1.0$ makes the distribution "sharper" (more confident/conservative).
      * Scaling logits by a number $> 1.0$ makes the distribution "flatter" (more random/creative).
4.  Sample from the distribution.
5.  Append the new token to the input.
6.  Repeat.

Implement the temperature scaling logic.



In [ ]:
def generate_text(model: tf.keras.Model, start_string: str,
                  temperature: float = 0.7, num_generate: int = 200) -> str:
    """
    Generates new text starting with the provided prompt.

    Args:
        model: The trained GPT model.
        start_string: The prompt.
        temperature: Controls randomness (0.0 to 1.0). Lower is more deterministic.
        num_generate: Number of tokens to generate.
    """
    # 1. Encode the prompt
    input_ids = tokenizer.encode(start_string).ids
    # Convert to tensor and add Batch dimension: (1, Time)
    input_eval = tf.convert_to_tensor([input_ids], dtype=tf.int32)

    generated_tokens: List[int] = []

    print(f"Generating {num_generate} tokens with temperature {temperature}...")

    for _ in range(num_generate):
        # 2. Forward Pass (training=False disables dropout)
        # predictions shape: (1, Time, Vocab_Size)
        predictions = model(input_eval, training=False)

        # 3. We only care about the prediction for the LAST token
        # logits shape: (1, Vocab_Size)
        logits = predictions[:, -1, :]

        # 4. Apply Temperature
        # --- YOUR CODE HERE ---
        #

        # -------------------------------

        # 5. Sample from the distribution
        # tf.random.categorical expects logits, returns index
        predicted_id = tf.random.categorical(logits, num_samples=1, dtype=tf.int32)

        # 6. Update the input for the next step
        # Append the predicted ID to the sequence
        input_eval = tf.concat([input_eval, predicted_id], axis=-1)

        # Crucial: Crop the sequence to BLOCK_SIZE if it gets too long.
        # The model cannot process sequences longer than it was trained on.
        if input_eval.shape[1] > BLOCK_SIZE:
             input_eval = input_eval[:, -BLOCK_SIZE:]

        # Save the result (convert tensor to int)
        generated_tokens.append(int(predicted_id[0, 0].numpy()))

    # 7. Decode the generated tokens back to text
    generated_text = tokenizer.decode(generated_tokens)
    return start_string + generated_text

# --- Run Inference ---
# Try different prompts and temperatures!
print("-" * 50)
result = generate_text(model, start_string="ROMEO: ", temperature=0.6)
print(result)
print("-" * 50)


# Question (5 Points)

In Causal Self-Attention, we applied a mask that sets values in the upper triangle of the attention matrix to negative infinity ($-10^9$).

  * **Why** is this physically necessary for a text generation model?
  * **What** would happen during training if we removed this mask?


# Bonus 1 [10 points]
Implement the unweighted regularized Matrix Factorization Collaborative Filtering algorithm to build a recommendation system.

# Bonus 2 [10 points]

Build a deep neural network using Tensorflow to implement a recommendation system using the same dataset.
You will be judged on the design choices you make (how many layers, embeddings, loss function etc), and whether you have a good rationale for making these choices. Please explain these in your notebook.
